<a href="https://colab.research.google.com/github/Jingxian12/movie-recommender-system/blob/main/Movie_Recommender_System_(PART_A)_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <div align =center > **TITLE : MOVIE RECOMMENDER SYSTEM**</div>

# **PART A : ETL**

**Summary:** I perform data preprocessing by integrating the `MovieLens` dataset with the `TMDB` dataset from **Kaggle**. For movies rated by users in MovieLens that are missing or incomplete in the Kaggle TMDB dataset, I use the **TMDB API** to extract the required missing metadata, ensuring a more complete and enriched movie information dataset to improve data completeness and support more effective recommendation modelling for our recommendation system later.


## **DATA ACQUISITION**

### - Import Necessary Libraries

In [ ]:
import pandas as pd
import numpy as np
import ast
import requests
import time
import json

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


We have 2 datasets to upload, which are MovieLens and TMDB Dataset

- `MovieLens` is for **Collaborative Filtering**
- `TMDB` is for **Content Based Filtering** and **NLP Based Filtering**

P.S. some of the tmdbId in `link.csv` have been modify myself because some movie have been removal or change the path in TMDB website(https://www.themoviedb.org/)

## **MovieLens**

In [ ]:
movies = pd.read_csv("/content/drive/MyDrive/dataset/movie/movieLens/movies.csv")
ratings = pd.read_csv("/content/drive/MyDrive/dataset/movie/movieLens/ratings.csv")
links = pd.read_csv("/content/drive/MyDrive/dataset/movie/movieLens/links.csv")

In [ ]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [ ]:
links.head()

,movieId,imdbId,tmdbId,media_type
0,1,114709,862,movie
1,2,113497,8844,movie
2,3,113228,15602,movie
3,4,114885,31357,movie
4,5,113041,11862,movie


In [ ]:
movieLens = pd.merge(
    movies,
    links,
    on="movieId",
    how="inner"
)

In [ ]:
movieLens.head()

,movieId,title,genres,imdbId,tmdbId,media_type
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862,movie
1,2,Jumanji (1995),Adventure|Children|Fantasy,113497,8844,movie
2,3,Grumpier Old Men (1995),Comedy|Romance,113228,15602,movie
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,114885,31357,movie
4,5,Father of the Bride Part II (1995),Comedy,113041,11862,movie


Let's check for duplicate rows in the `movieLens` DataFrame based on the `tmdbId` column.

In [ ]:
# Check column duplicated
duplicate_tmdb_ids_rows = movieLens[movieLens.duplicated(subset=['tmdbId','media_type'], keep=False)].sort_values('title')
display(duplicate_tmdb_ids_rows)

print(f"Number of duplicate rows based on 'tmdbId': {len(duplicate_tmdb_ids_rows)}")

,movieId,title,genres,imdbId,tmdbId,media_type
4169,6003,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Thriller,290538,4912,movie
9106,144606,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Romance|Thriller,270288,4912,movie
5854,32600,Eros (2004),Drama,377059,39850,movie
9135,147002,Eros (2004),Drama|Romance,343663,39850,movie
2141,2851,Saturn 3 (1980),Adventure|Sci-Fi|Thriller,81454,19761,movie
9468,168358,Saturn 3 (1980),Sci-Fi|Thriller,79285,19761,movie


Number of duplicate rows based on 'tmdbId': 6


I manually checked the **IDs** and noticed that these three movies are duplicates because **newer versions were updated** in the MovieLens dataset, so I kept the rows with the correct `MovieLens IDs (movieId)`,`IMDb IDs (imdbId)`,  and `TMDb IDs (tmdbId)`.

In [ ]:
movie_ids_to_remove = [6003, 32600, 2851]
movieLens = movieLens[~movieLens['movieId'].isin(movie_ids_to_remove)] # ~ means NOT

print(f"Updated movieLens DataFrame shape: {movieLens.shape}")

Updated movieLens DataFrame shape: (9739, 6)


At the same time, I standardized the ratings data by mapping duplicate MovieLens movie IDs to their correct movie IDs, ensuring all ratings for the same movie are consolidated under a single consistent identifier.

In [ ]:
movie_id_mapping = {
    6003: 144606,
    32600: 147002,
    2851: 168358
}

ratings['movieId'] = ratings['movieId'].replace(movie_id_mapping)

print("Movie IDs in ratings DataFrame have been standardized.")

Movie IDs in ratings DataFrame have been standardized.


In [ ]:
# Verify by checking the counts of these movie IDs or sampling the ratings DataFrame
print(ratings[ratings['movieId'].isin([6003,32600, 2851])].head())

Empty DataFrame
Columns: [userId, movieId, rating, timestamp]
Index: []


In [ ]:
ratings.shape

(100836, 4)

## **TMDB Dataset**

In [ ]:
tmdb_movies = pd.read_csv("/content/drive/MyDrive/dataset/movie/tmdb/tmdb_5000_movies.csv")
tmdb_credits = pd.read_csv("/content/drive/MyDrive/dataset/movie/tmdb/tmdb_5000_credits.csv")

In [ ]:
tmdb = tmdb_movies.merge(
    tmdb_credits,
    left_on="id",
    right_on="movie_id",
)

In [ ]:
tmdb.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,spoken_languages,status,tagline,title_x,vote_average,vote_count,movie_id,title_y,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...",...,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...",...,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...",...,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]",...,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [ ]:
tmdb.shape

(4803, 24)

In [ ]:
tmdb.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   object 
 2   homepage              1712 non-null   object 
 3   id                    4803 non-null   int64  
 4   keywords              4803 non-null   object 
 5   original_language     4803 non-null   object 
 6   original_title        4803 non-null   object 
 7   overview              4800 non-null   object 
 8   popularity            4803 non-null   float64
 9   production_companies  4803 non-null   object 
 10  production_countries  4803 non-null   object 
 11  release_date          4802 non-null   object 
 12  revenue               4803 non-null   int64  
 13  runtime               4801 non-null   float64
 14  spoken_languages      4803 non-null   object 
 15  status               

In [ ]:
tmdb =tmdb[['movie_id','title_x','genres','keywords','original_language','overview','production_companies','release_date','runtime','popularity','vote_average','vote_count','cast','crew']]

In [ ]:
tmdb.head()

,movie_id,title_x,genres,keywords,original_language,overview,production_companies,release_date,runtime,popularity,vote_average,vote_count,cast,crew
0,19995,Avatar,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,"In the 22nd century, a paraplegic Marine is di...","[{""name"": ""Ingenious Film Partners"", ""id"": 289...",2009-12-10,162.0,150.437577,7.2,11800,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,"Captain Barbossa, long believed to be dead, ha...","[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...",2007-05-19,169.0,139.082615,6.9,4500,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,A cryptic message from Bond’s past sends him o...,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...",2015-10-26,148.0,107.376788,6.3,4466,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,Following the death of District Attorney Harve...,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...",2012-07-16,165.0,112.312950,7.6,9106,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,"John Carter is a war-weary, former military ca...","[{""name"": ""Walt Disney Pictures"", ""id"": 2}]",2012-03-07,132.0,43.926995,6.1,2124,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [ ]:
print(tmdb.shape)
print(tmdb.info())

(4803, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   movie_id              4803 non-null   int64  
 1   title_x               4803 non-null   object 
 2   genres                4803 non-null   object 
 3   keywords              4803 non-null   object 
 4   original_language     4803 non-null   object 
 5   overview              4800 non-null   object 
 6   production_companies  4803 non-null   object 
 7   release_date          4802 non-null   object 
 8   runtime               4801 non-null   float64
 9   popularity            4803 non-null   float64
 10  vote_average          4803 non-null   float64
 11  vote_count            4803 non-null   int64  
 12  cast                  4803 non-null   object 
 13  crew                  4803 non-null   object 
dtypes: float64(3), int64(2), object(9)
memory usage: 525.5+ KB
No

### **- Check Missing Values**

In [ ]:
tmdb.isnull().sum()

,0
movie_id,0
title_x,0
genres,0
keywords,0
original_language,0
overview,3
production_companies,0
release_date,1
runtime,2
popularity,0


In [ ]:
missing_info = tmdb[tmdb.isnull().any(axis=1)]
missing_info

,movie_id,title_x,genres,keywords,original_language,overview,production_companies,release_date,runtime,popularity,vote_average,vote_count,cast,crew
2656,370980,Chiamatemi Francesco - Il Papa della gente,"[{""id"": 18, ""name"": ""Drama""}]","[{""id"": 717, ""name"": ""pope""}, {""id"": 5565, ""na...",it,NaN,"[{""name"": ""Taodue Film"", ""id"": 45724}]",2015-12-03,NaN,0.738646,7.3,12,"[{""cast_id"": 5, ""character"": ""Jorge Mario Berg...","[{""credit_id"": ""5660019ac3a36875f100252b"", ""de..."
4140,459488,"To Be Frank, Sinatra at 100","[{""id"": 99, ""name"": ""Documentary""}]","[{""id"": 6027, ""name"": ""music""}, {""id"": 225822,...",en,NaN,"[{""name"": ""Eyeline Entertainment"", ""id"": 60343}]",2015-12-12,NaN,0.050625,0.0,0,"[{""cast_id"": 0, ""character"": ""Narrator"", ""cred...","[{""credit_id"": ""592b25e4c3a368783e065a2f"", ""de..."
4431,292539,Food Chains,"[{""id"": 99, ""name"": ""Documentary""}]",[],de,NaN,[],2014-04-26,83.0,0.795698,7.4,8,[],"[{""credit_id"": ""5470c3b1c3a368085e000abd"", ""de..."
4553,380097,America Is Still the Place,[],[],en,1971 post civil rights San Francisco seemed li...,[],NaN,0.0,0.000000,0.0,0,[],[]


In [ ]:
tmdb = tmdb.dropna()
tmdb.shape

(4799, 14)

I removed rows with missing values from this dataset `(TMDb 5000 Movie Dataset from Kaggle)` because the dataset **might not fully match the movies rated by users in the MovieLens dataset**,

If the removed movies are useful for users, more **comprehensive information** can still be retrieved through the `TMDB API`.

### **- Preprocessing for TMDB dataset**

In [ ]:
tmdb.head()

,movie_id,title_x,genres,keywords,original_language,overview,production_companies,release_date,runtime,popularity,vote_average,vote_count,cast,crew
0,19995,Avatar,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,"In the 22nd century, a paraplegic Marine is di...","[{""name"": ""Ingenious Film Partners"", ""id"": 289...",2009-12-10,162.0,150.437577,7.2,11800,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,"Captain Barbossa, long believed to be dead, ha...","[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...",2007-05-19,169.0,139.082615,6.9,4500,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,A cryptic message from Bond’s past sends him o...,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...",2015-10-26,148.0,107.376788,6.3,4466,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,Following the death of District Attorney Harve...,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...",2012-07-16,165.0,112.312950,7.6,9106,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,"John Carter is a war-weary, former military ca...","[{""name"": ""Walt Disney Pictures"", ""id"": 2}]",2012-03-07,132.0,43.926995,6.1,2124,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


From the dataframe, several columns such as `genres`, `keywords`, `production_companies`, `cast`, and `crew` are **stored as nested list/dictionary structures**, so I will standardize these fields by extracting only the key values to make the data more consistent and readable.

In [ ]:
# General converter(to get name from list)
def converter(x):
    if isinstance(x, str):
        x = ast.literal_eval(x)

    return [i["name"] for i in x] if x else []

#### Genres

In [ ]:
tmdb['genres'][0]

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

In [ ]:
tmdb["genres"] = tmdb["genres"].apply(converter)

In [ ]:
tmdb['genres'].head()

,genres
0,"[Action, Adventure, Fantasy, Science Fiction]"
1,"[Adventure, Fantasy, Action]"
2,"[Action, Adventure, Crime]"
3,"[Action, Crime, Drama, Thriller]"
4,"[Action, Adventure, Science Fiction]"


#### Keywords

In [ ]:
tmdb['keywords'][0]

'[{"id": 1463, "name": "culture clash"}, {"id": 2964, "name": "future"}, {"id": 3386, "name": "space war"}, {"id": 3388, "name": "space colony"}, {"id": 3679, "name": "society"}, {"id": 3801, "name": "space travel"}, {"id": 9685, "name": "futuristic"}, {"id": 9840, "name": "romance"}, {"id": 9882, "name": "space"}, {"id": 9951, "name": "alien"}, {"id": 10148, "name": "tribe"}, {"id": 10158, "name": "alien planet"}, {"id": 10987, "name": "cgi"}, {"id": 11399, "name": "marine"}, {"id": 13065, "name": "soldier"}, {"id": 14643, "name": "battle"}, {"id": 14720, "name": "love affair"}, {"id": 165431, "name": "anti war"}, {"id": 193554, "name": "power relations"}, {"id": 206690, "name": "mind and soul"}, {"id": 209714, "name": "3d"}]'

In [ ]:
tmdb["keywords"] = tmdb["keywords"].apply(converter)

In [ ]:
tmdb['keywords'].head()

,keywords
0,"[culture clash, future, space war, space colon..."
1,"[ocean, drug abuse, exotic island, east india ..."
2,"[spy, based on novel, secret agent, sequel, mi..."
3,"[dc comics, crime fighter, terrorist, secret i..."
4,"[based on novel, mars, medallion, space travel..."


#### Production Comapnies

In [ ]:
tmdb['production_companies'][0]

'[{"name": "Ingenious Film Partners", "id": 289}, {"name": "Twentieth Century Fox Film Corporation", "id": 306}, {"name": "Dune Entertainment", "id": 444}, {"name": "Lightstorm Entertainment", "id": 574}]'

In [ ]:
tmdb["production_companies"] = tmdb["production_companies"].apply(converter)

In [ ]:
tmdb['production_companies'].head()

,production_companies
0,"[Ingenious Film Partners, Twentieth Century Fo..."
1,"[Walt Disney Pictures, Jerry Bruckheimer Films..."
2,"[Columbia Pictures, Danjaq, B24]"
3,"[Legendary Pictures, Warner Bros., DC Entertai..."
4,[Walt Disney Pictures]


#### Cast

In [ ]:
def extract_cast_info(cast):
    if isinstance(cast, str):
        cast = ast.literal_eval(cast)

    cast_info = [
        {
            "character": c.get("character"),
            "name": c.get("name")
        }
        for c in cast
    ]

    return cast_info

In [ ]:
tmdb['cast'][0]

'[{"cast_id": 242, "character": "Jake Sully", "credit_id": "5602a8a7c3a3685532001c9a", "gender": 2, "id": 65731, "name": "Sam Worthington", "order": 0}, {"cast_id": 3, "character": "Neytiri", "credit_id": "52fe48009251416c750ac9cb", "gender": 1, "id": 8691, "name": "Zoe Saldana", "order": 1}, {"cast_id": 25, "character": "Dr. Grace Augustine", "credit_id": "52fe48009251416c750aca39", "gender": 1, "id": 10205, "name": "Sigourney Weaver", "order": 2}, {"cast_id": 4, "character": "Col. Quaritch", "credit_id": "52fe48009251416c750ac9cf", "gender": 2, "id": 32747, "name": "Stephen Lang", "order": 3}, {"cast_id": 5, "character": "Trudy Chacon", "credit_id": "52fe48009251416c750ac9d3", "gender": 1, "id": 17647, "name": "Michelle Rodriguez", "order": 4}, {"cast_id": 8, "character": "Selfridge", "credit_id": "52fe48009251416c750ac9e1", "gender": 2, "id": 1771, "name": "Giovanni Ribisi", "order": 5}, {"cast_id": 7, "character": "Norm Spellman", "credit_id": "52fe48009251416c750ac9dd", "gender": 

In [ ]:
tmdb["cast"] = tmdb["cast"].apply(extract_cast_info)

In [ ]:
tmdb["cast"].head()

,cast
0,"[{'character': 'Jake Sully', 'name': 'Sam Wort..."
1,"[{'character': 'Captain Jack Sparrow', 'name':..."
2,"[{'character': 'James Bond', 'name': 'Daniel C..."
3,"[{'character': 'Bruce Wayne / Batman', 'name':..."
4,"[{'character': 'John Carter', 'name': 'Taylor ..."


#### Crew

In [ ]:
def extract_director(crew):
    if isinstance(crew, str):
        crew = ast.literal_eval(crew)

    directors = [c["name"] for c in crew if c.get("job") == "Director"]
    return directors

In [ ]:
tmdb['crew'][0]

'[{"credit_id": "52fe48009251416c750aca23", "department": "Editing", "gender": 0, "id": 1721, "job": "Editor", "name": "Stephen E. Rivkin"}, {"credit_id": "539c47ecc3a36810e3001f87", "department": "Art", "gender": 2, "id": 496, "job": "Production Design", "name": "Rick Carter"}, {"credit_id": "54491c89c3a3680fb4001cf7", "department": "Sound", "gender": 0, "id": 900, "job": "Sound Designer", "name": "Christopher Boyes"}, {"credit_id": "54491cb70e0a267480001bd0", "department": "Sound", "gender": 0, "id": 900, "job": "Supervising Sound Editor", "name": "Christopher Boyes"}, {"credit_id": "539c4a4cc3a36810c9002101", "department": "Production", "gender": 1, "id": 1262, "job": "Casting", "name": "Mali Finn"}, {"credit_id": "5544ee3b925141499f0008fc", "department": "Sound", "gender": 2, "id": 1729, "job": "Original Music Composer", "name": "James Horner"}, {"credit_id": "52fe48009251416c750ac9c3", "department": "Directing", "gender": 2, "id": 2710, "job": "Director", "name": "James Cameron"},

In [ ]:
tmdb["director"] = tmdb["crew"].apply(extract_director)

In [ ]:
tmdb['director'].head()

,director
0,[James Cameron]
1,[Gore Verbinski]
2,[Sam Mendes]
3,[Christopher Nolan]
4,[Andrew Stanton]


### **- Final TMDB Dataset**


In [ ]:
tmdb.head()

,movie_id,title_x,genres,keywords,original_language,overview,production_companies,release_date,runtime,popularity,vote_average,vote_count,cast,crew,director
0,19995,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...",en,"In the 22nd century, a paraplegic Marine is di...","[Ingenious Film Partners, Twentieth Century Fo...",2009-12-10,162.0,150.437577,7.2,11800,"[{'character': 'Jake Sully', 'name': 'Sam Wort...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",[James Cameron]
1,285,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...",en,"Captain Barbossa, long believed to be dead, ha...","[Walt Disney Pictures, Jerry Bruckheimer Films...",2007-05-19,169.0,139.082615,6.9,4500,"[{'character': 'Captain Jack Sparrow', 'name':...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...",[Gore Verbinski]
2,206647,Spectre,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...",en,A cryptic message from Bond’s past sends him o...,"[Columbia Pictures, Danjaq, B24]",2015-10-26,148.0,107.376788,6.3,4466,"[{'character': 'James Bond', 'name': 'Daniel C...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de...",[Sam Mendes]
3,49026,The Dark Knight Rises,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...",en,Following the death of District Attorney Harve...,"[Legendary Pictures, Warner Bros., DC Entertai...",2012-07-16,165.0,112.312950,7.6,9106,"[{'character': 'Bruce Wayne / Batman', 'name':...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de...",[Christopher Nolan]
4,49529,John Carter,"[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...",en,"John Carter is a war-weary, former military ca...",[Walt Disney Pictures],2012-03-07,132.0,43.926995,6.1,2124,"[{'character': 'John Carter', 'name': 'Taylor ...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de...",[Andrew Stanton]


In [ ]:
tmdb = tmdb.drop(columns=['crew'])

In [ ]:
tmdb = tmdb.rename(columns={"title_x": "title"})

In [ ]:
cols = ["genres", "keywords","production_companies","cast", "director"]

for col in cols:
    tmdb[col] = tmdb[col].apply(lambda x: np.nan if x == [] else x)

In [ ]:
tmdb[cols].isnull().sum()

,0
genres,27
keywords,410
production_companies,349
cast,41
director,29


In [ ]:
tmdb = tmdb.dropna(subset=["genres","cast", "director"])

In [ ]:
tmdb.isnull().sum()

,0
movie_id,0
title,0
genres,0
keywords,363
original_language,0
overview,0
production_companies,300
release_date,0
runtime,0
popularity,0


I did not remove the `keywords` and `production_companies` features because it contains a large number of values (around 400+ missing entries), and removing or reprocessing it would be **costly**, while it is also not a critical feature for content-based filtering compared to other attributes like genres or cast

### **- Convert back to empty list to ensure consistency after api extraction and merged dataset**

In [ ]:
tmdb.loc[:, 'keywords'] = tmdb['keywords'].apply(lambda x: x if isinstance(x, list) else [])
tmdb.loc[:, 'production_companies'] = tmdb['production_companies'].apply(lambda x: x if isinstance(x, list) else [])

In [ ]:
tmdb.isnull().sum()

,0
movie_id,0
title,0
genres,0
keywords,0
original_language,0
overview,0
production_companies,0
release_date,0
runtime,0
popularity,0


## **Combined Dataset**
- TMDB Dataset: Add media type and source columns
- Missing_movies datset : Add source columns



### **- Find how many movies are missing at tmdb dataset (Based on movieLens movie)**


In [ ]:
# TMDB Movie
tmdb_ids = set(tmdb['movie_id'])

# MovieLens Movie
ml_ids = set(movieLens['tmdbId'])

missing_tmdb_ids = ml_ids - tmdb_ids

# Movies that does not have deeper information
missing_movies = movieLens[movieLens['tmdbId'].isin(missing_tmdb_ids)]

In [ ]:
missing_movies.head()

,movieId,title,genres,imdbId,tmdbId,media_type
1,2,Jumanji (1995),Adventure|Children|Fantasy,113497,8844,movie
2,3,Grumpier Old Men (1995),Comedy|Romance,113228,15602,movie
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,114885,31357,movie
4,5,Father of the Bride Part II (1995),Comedy,113041,11862,movie
5,6,Heat (1995),Action|Crime|Thriller,113277,949,movie


In [ ]:
print(missing_movies.isna().sum())
print(missing_movies.shape)

movieId       0
title         0
genres        0
imdbId        0
tmdbId        0
media_type    0
dtype: int64
(6198, 6)


Here have around 6200 movies that have been rated by users, but does not have deeper information (such as cast, character, popularity, release date and so on)



So that, I combined the `TMDB 5000 dataset` from **Kaggle** with `MovieLens user-rated movies` that were missing from the TMDB 5000 dataset (**missing_movies dataframe)**, allowing missing movie information to be retrieved through the TMDB API more efficiently.

In [ ]:
missing_movies = missing_movies.drop(columns=['movieId','title','genres','imdbId'])
missing_movies["source"] = "movielens"

In [ ]:
tmdb['media_type'] = 'movie'
tmdb['source'] = 'kaggle'

In [ ]:
print(tmdb.shape)
print(missing_movies.shape)

(4738, 16)
(6198, 3)


In [ ]:
tmdb = tmdb.rename(columns={"movie_id":"tmdbId"})
combined_df = pd.concat([tmdb, missing_movies], ignore_index=True)

In [ ]:
combined_df.shape

(10936, 16)

In [ ]:
combine_duplicate = combined_df[combined_df.duplicated(subset=['tmdbId','media_type'],keep=False)]
combine_duplicate

,tmdbId,title,genres,keywords,original_language,overview,production_companies,release_date,runtime,popularity,vote_average,vote_count,cast,director,media_type,source


In [ ]:
combined_df.head()

,tmdbId,title,genres,keywords,original_language,overview,production_companies,release_date,runtime,popularity,vote_average,vote_count,cast,director,media_type,source
0,19995,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...",en,"In the 22nd century, a paraplegic Marine is di...","[Ingenious Film Partners, Twentieth Century Fo...",2009-12-10,162.0,150.437577,7.2,11800.0,"[{'character': 'Jake Sully', 'name': 'Sam Wort...",[James Cameron],movie,kaggle
1,285,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...",en,"Captain Barbossa, long believed to be dead, ha...","[Walt Disney Pictures, Jerry Bruckheimer Films...",2007-05-19,169.0,139.082615,6.9,4500.0,"[{'character': 'Captain Jack Sparrow', 'name':...",[Gore Verbinski],movie,kaggle
2,206647,Spectre,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...",en,A cryptic message from Bond’s past sends him o...,"[Columbia Pictures, Danjaq, B24]",2015-10-26,148.0,107.376788,6.3,4466.0,"[{'character': 'James Bond', 'name': 'Daniel C...",[Sam Mendes],movie,kaggle
3,49026,The Dark Knight Rises,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...",en,Following the death of District Attorney Harve...,"[Legendary Pictures, Warner Bros., DC Entertai...",2012-07-16,165.0,112.312950,7.6,9106.0,"[{'character': 'Bruce Wayne / Batman', 'name':...",[Christopher Nolan],movie,kaggle
4,49529,John Carter,"[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...",en,"John Carter is a war-weary, former military ca...",[Walt Disney Pictures],2012-03-07,132.0,43.926995,6.1,2124.0,"[{'character': 'John Carter', 'name': 'Taylor ...",[Andrew Stanton],movie,kaggle


I have add a new column called `source` for both datasets to improve the efficiency of the API extraction process. This column is used to distinguish between records originating from the Kaggle TMDB dataset and those from the MovieLens dataset.

- The **Kaggle** dataset already contains most of the required movie metadata, except for poster-related information, whereas the **MovieLens** dataset lacks nearly all detailed movie information.

By introducing the `source` column, the data processing pipeline can apply different extraction strategies: **lightweight API calls for Kaggle records to retrieve only poster data**, and **full metadata enrichment via the API for MovieLens records**. This separation helps optimize API usage, reduce unnecessary requests, and improve overall processing efficiency.

## **API**

### **Extraction Using API KEY**

In [ ]:
from tqdm import tqdm  #customizable progress bars to loops

# =========================================================
# CONFIG
# =========================================================
API_KEY = "API_KEY"
CACHE_FILE = "/content/drive/MyDrive/dataset/movie/API/tmdb_cache1.json"
BASE_URL = "https://api.themoviedb.org/3"
RATE_LIMIT_SLEEP = 0.25

# =========================================================
# LOAD CACHE
# =========================================================
try:
    with open(CACHE_FILE, "r") as f:
        cache = json.load(f)

except FileNotFoundError:
    cache = {}

print(f"✅ Loaded cache items: {len(cache)}")

✅ Loaded cache items: 0


In [ ]:
# 2 Functions : request_json and build_tmdb_response

# =========================================================
# SAFE REQUEST
# =========================================================
def request_json(url):
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            return response.json()
        return None
    except Exception:
        return None

# =========================================================
# EXTRACT MOVIE OR TV
# =========================================================
def build_tmdb_response(details, media_type):
        credits = details.get("credits", {})
        keywords = details.get("keywords", {})

        # TV uses "name" instead of "title"
        title = (
            details.get("title")
            if media_type == "movie"
            else details.get("name")
        )
        # TV uses "first_air_date"
        release_date = (
            details.get("release_date")
            if media_type == "movie"
            else details.get("first_air_date")
        )
        # TV uses episode runtime list
        runtime = (
            details.get("runtime")
            if media_type == "movie"
            else (
                details.get("episode_run_time", [None])[0]
                if details.get("episode_run_time")
                else None
            )
        )
        # keyword structure differs
        keyword_source = (
            keywords.get("keywords", [])
            if media_type == "movie"
            else keywords.get("results", [])
        )
        keyword_list = [
            k.get("name")
            for k in keyword_source
            if k.get("name")
        ]
        # Poster related
        poster_path = details.get("poster_path")

        # Return Values
        return {
            "tmdbId": details.get("id"),
            "media_type": media_type,
            "title": title,

            "genres": [
                g.get("name")
                for g in details.get("genres", [])
                if g.get("name")
            ],

            "keywords": keyword_list,
            "original_language": details.get("original_language"),
            "overview": details.get("overview"),

            "production_companies": [
                p.get("name")
                for p in details.get("production_companies", [])
                if p.get("name")
            ],

            "release_date": release_date,
            "runtime": runtime,
            "popularity": details.get("popularity"),
            "vote_average": details.get("vote_average"),
            "vote_count": details.get("vote_count"),

            "cast": [
                {
                    "character": c.get("character") if c.get("character") else "Unknown Character",
                    "actor": c.get("name") if c.get("name") else "Unknown Actor"
                }
                for c in credits.get("cast", [])
                if c.get("name") or c.get("character")
            ],

            "director": [
                c.get("name")
                for c in credits.get("crew", [])
                if c.get("job") == "Director" and c.get("name")
            ],

            "poster_path": poster_path,

            "poster_url": (
                f"https://image.tmdb.org/t/p/w500{poster_path}"
                if poster_path
                else None
            )
        }

In [ ]:
# =========================================================
#  A) KAGGLE
# =========================================================
# GET POSTER only
def fetch_tmdb_poster(tmdb_id, media_type):
    try:
        url = f"{BASE_URL}/{media_type}/{tmdb_id}?api_key={API_KEY}"
        details = request_json(url)

        if details is None:
            return None

        poster_path = details.get("poster_path")

        return {
            "poster_path": poster_path,
            "poster_url": (
                f"https://image.tmdb.org/t/p/w500{poster_path}"
                if poster_path
                else None
            )
        }

    except Exception as e:
        print(f"❌ Poster error {tmdb_id}: {e}")
        return None

# =========================================================
#  B) MOVIELENS
# =========================================================

# GET MOVIE
def fetch_tmdb_movie(tmdb_id):
    try:
        url = (
            f"{BASE_URL}/movie/{tmdb_id}"
            f"?api_key={API_KEY}"
            f"&append_to_response=credits,keywords"
        )
        details = request_json(url)
        if details is None:
            return None

        return build_tmdb_response(details, "movie")

    except Exception as e:
        print(f"❌ Movie error {tmdb_id}: {e}")
        return None

# GET TV
def fetch_tmdb_tv(tmdb_id):
    try:
        url = (
            f"{BASE_URL}/tv/{tmdb_id}"
            f"?api_key={API_KEY}"
            f"&append_to_response=credits,keywords"
        )
        details = request_json(url)
        if details is None:
            return None

        return build_tmdb_response(details, "tv")

    except Exception as e:
        print(f"❌ TV error {tmdb_id}: {e}")
        return None

In [ ]:
import json
import time
import math
import pandas as pd
import numpy as np
from tqdm import tqdm

STATUS_SUCCESS = "success"
STATUS_FAIL = "failed"

# -----------------------------
# SAFE blank checker
# -----------------------------
def is_blank(v):
    if v is None:
        return True

    # numpy array / list
    if isinstance(v, (list, tuple, np.ndarray)):
        return len(v) == 0

    # pandas / numpy NaN
    try:
        if pd.isna(v):
            return True
    except:
        pass

    return v == ""


# -----------------------------
# CLEAN pandas row → python safe
# -----------------------------
def clean_row(d):
    cleaned = {}

    for k, v in d.items():

        # convert numpy arrays → list
        if isinstance(v, np.ndarray):
            v = v.tolist()

        # convert NaN → None
        try:
            if pd.isna(v):
                v = None
        except:
            pass

        cleaned[k] = v

    return cleaned


# =========================================================
# MAIN FUNCTION
# =========================================================
def extract_tmdb_data(df, cache):

    df = df.copy()
    rows = df.dropna(subset=["tmdbId"])

    # ---------------------------------
    # CACHE INIT
    # ---------------------------------
    cache.setdefault("tmdb_cache", {})
    tmdb_cache = cache["tmdb_cache"]

    api_results = []

    # ---------------------------------
    # LOOP
    # ---------------------------------
    for idx, row in enumerate(
        tqdm(rows.itertuples(index=False), total=len(rows)),
        start=1
    ):

        tmdb_id = str(int(row.tmdbId))
        media_type = row.media_type
        source = row.source

        cache_key = f"{media_type}_{tmdb_id}"

        data = None
        poster_status = None
        status = None

        # =====================================================
        # CACHE CHECK
        # =====================================================
        cached = tmdb_cache.get(cache_key)

        if cached:
            if cached.get("status") == STATUS_FAIL:
                continue

            data = cached.get("data")
            poster_status = cached.get("poster_status")
            status = cached.get("status")

        # =====================================================
        # FETCH
        # =====================================================
        if data is None:

            try:

                # -----------------------------
                # KAGGLE
                # -----------------------------
                if source == "kaggle":

                    base_data = clean_row(row._asdict())

                    poster_data = fetch_tmdb_poster(tmdb_id, media_type) or {}
                    poster_data = clean_row(poster_data)

                    data = {
                        **base_data,
                        **poster_data
                    }

                    poster_path = poster_data.get("poster_path")
                    poster_url = poster_data.get("poster_url")

                    poster_status = (
                        STATUS_SUCCESS
                        if not is_blank(poster_path)
                        and not is_blank(poster_url)
                        else STATUS_FAIL
                    )

                    status = (
                        STATUS_SUCCESS
                        if poster_status == STATUS_SUCCESS
                        else STATUS_FAIL
                    )

                # -----------------------------
                # MOVIELENS
                # -----------------------------
                else:

                    if media_type == "movie":
                        data = fetch_tmdb_movie(tmdb_id)

                    elif media_type == "tv":
                        data = fetch_tmdb_tv(tmdb_id)

                    data = clean_row(data) if data else None

                    status = (
                        STATUS_SUCCESS
                        if data
                        else STATUS_FAIL
                    )

            except Exception as e:

                print(f"❌ Error {cache_key}: {e}")

                if source == "kaggle":
                    data = clean_row(row._asdict())
                    poster_status = STATUS_FAIL
                    status = STATUS_FAIL
                else:
                    data = None
                    status = STATUS_FAIL

            # =====================================================
            # CACHE SAVE
            # =====================================================
            tmdb_cache[cache_key] = {
                "status": status,
                "poster_status": poster_status,
                "data": data
            }

            time.sleep(RATE_LIMIT_SLEEP)

        # =====================================================
        # OUTPUT
        # =====================================================
        if data:

            result_item = {
                "cache_key": cache_key,
                "status": status,
                "poster_status": poster_status,
                "media_type": media_type,
                "source": source,
                "tmdbId": tmdb_id,
                **data
            }

        else:

            result_item = {
                "cache_key": cache_key,
                "status": STATUS_FAIL,
                "poster_status": poster_status,
                "media_type": media_type,
                "source": source,
                "tmdbId": tmdb_id,
                "title": None,
                "overview": None,
                "cast": []
            }

        api_results.append(result_item)

        # periodic save
        if idx % 50 == 0:
            with open(CACHE_FILE, "w") as f:
                json.dump(cache, f)

            print(f"💾 Saved | cache={len(tmdb_cache)}")

    # final save
    with open(CACHE_FILE, "w") as f:
        json.dump(cache, f)

    print("\n✅ DONE")
    print(f"Results: {len(api_results)}")
    print(f"Cache size: {len(tmdb_cache)}")

    return api_results

In [ ]:
api_results = extract_tmdb_data(combined_df, cache)

  0%|          | 50/10936 [00:16<1:00:20,  3.01it/s]

💾 Saved | cache=50


  1%|          | 71/10936 [00:22<57:01,  3.18it/s]/tmp/ipykernel_1287/2796670177.py:46: DeprecationWarning: The truth value of an empty array is ambiguous. Returning False, but in future this will result in an error. Use `array.size > 0` to check that an array is not empty.
  if pd.isna(v):
  1%|          | 100/10936 [00:32<1:02:09,  2.91it/s]

💾 Saved | cache=100


  1%|▏         | 150/10936 [00:48<1:00:03,  2.99it/s]

...

💾 Saved | cache=10850


100%|█████████▉| 10900/10936 [1:03:26<00:33,  1.06it/s]

💾 Saved | cache=10900


100%|██████████| 10936/10936 [1:03:38<00:00,  2.86it/s]



✅ DONE
Results: 10936
Cache size: 10936


### **Metrics Check**

#### - Failed Cache Keys Check

This function retrieves the cache keys of movies that failed the API extraction process, allowing us to identify IDs that may have been removed or deleted from the website.

In [ ]:
# =========================================================
# FAILED CACHE KEYS
# =========================================================
def get_failed_cache_keys(cache):
    tmdb_cache = cache.get("tmdb_cache", {})
    failed = [
        k for k, v in tmdb_cache.items()
        if v.get("status") == STATUS_FAIL
    ]
    print("❌ FAILED CACHE KEYS")
    print(f"Total failed: {len(failed)}")
    return failed

In [ ]:
get_failed_cache_keys(cache)

❌ FAILED CACHE KEYS
Total failed: 31


['movie_82525',
 'movie_10398',
 'movie_310706',
 'movie_112430',
 'movie_333355',
 'movie_153397',
 'movie_36597',
 'movie_183894',
 'movie_11042',
 'movie_395766',
 'movie_50942',
 'movie_364083',
 'movie_299553',
 'movie_146882',
 'movie_146269',
 'movie_181940',
 'movie_274758',
 'movie_361398',
 'movie_266857',
 'movie_215993',
 'movie_12224',
 'movie_54507',
 'movie_59017',
 'movie_202043',
 'movie_263947',
 'movie_215999',
 'movie_120605',
 'movie_327029',
 'movie_185789',
 'movie_65595',
 'movie_412103']

#### - Cache Report

This function generates a comprehensive cache report by counting total keys and categorizing them into successful, failed, or missing data status.

In [ ]:
# =========================================================
# CACHE HEALTH REPORT
# =========================================================
def cache_report(cache):
    tmdb_cache = cache.get("tmdb_cache", {})

    total = len(tmdb_cache)
    success = 0
    failed = 0
    missing = 0

    for v in tmdb_cache.values():
        if v.get("status") == STATUS_FAIL:
            failed += 1
        elif v.get("data") is None:
            missing += 1
        else:
            success += 1

    print("\n📊 CACHE REPORT")
    print(f"Total keys   : {total}")
    print(f"Success      : {success}")
    print(f"Failed       : {failed}")
    print(f"Missing data : {missing}")

    return {
        "total": total,
        "success": success,
        "failed": failed,
        "missing": missing
    }

In [ ]:
cache_report(cache)


📊 CACHE REPORT
Total keys   : 10936
Success      : 10905
Failed       : 31
Missing data : 0


{'total': 10936, 'success': 10905, 'failed': 31, 'missing': 0}

### **Full extraction Dataframe**


In [ ]:
final_tmdb_df = pd.DataFrame(api_results)
display(final_tmdb_df.head())

print(final_tmdb_df.shape)

,cache_key,status,poster_status,media_type,source,tmdbId,title,genres,keywords,original_language,...,production_companies,release_date,runtime,popularity,vote_average,vote_count,cast,director,poster_path,poster_url
0,movie_19995,success,success,movie,kaggle,19995,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...",en,...,"[Ingenious Film Partners, Twentieth Century Fo...",2009-12-10,162.0,150.437577,7.2,11800.0,"[{'character': 'Jake Sully', 'name': 'Sam Wort...",[James Cameron],/gKY6q7SjCkAU6FqvqWybDYgUKIF.jpg,https://image.tmdb.org/t/p/w500/gKY6q7SjCkAU6F...
1,movie_285,success,success,movie,kaggle,285,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...",en,...,"[Walt Disney Pictures, Jerry Bruckheimer Films...",2007-05-19,169.0,139.082615,6.9,4500.0,"[{'character': 'Captain Jack Sparrow', 'name':...",[Gore Verbinski],/jGWpG4YhpQwVmjyHEGkxEkeRf0S.jpg,https://image.tmdb.org/t/p/w500/jGWpG4YhpQwVmj...
2,movie_206647,success,success,movie,kaggle,206647,Spectre,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...",en,...,"[Columbia Pictures, Danjaq, B24]",2015-10-26,148.0,107.376788,6.3,4466.0,"[{'character': 'James Bond', 'name': 'Daniel C...",[Sam Mendes],/zj8ongFhtWNsVlfjOGo8pSr7PQg.jpg,https://image.tmdb.org/t/p/w500/zj8ongFhtWNsVl...
3,movie_49026,success,success,movie,kaggle,49026,The Dark Knight Rises,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...",en,...,"[Legendary Pictures, Warner Bros., DC Entertai...",2012-07-16,165.0,112.312950,7.6,9106.0,"[{'character': 'Bruce Wayne / Batman', 'name':...",[Christopher Nolan],/hr0L2aueqlP2BYUblTTjmtn0hw4.jpg,https://image.tmdb.org/t/p/w500/hr0L2aueqlP2BY...
4,movie_49529,success,success,movie,kaggle,49529,John Carter,"[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...",en,...,[Walt Disney Pictures],2012-03-07,132.0,43.926995,6.1,2124.0,"[{'character': 'John Carter', 'name': 'Taylor ...",[Andrew Stanton],/lCxz1Yus07QCQQCb6I0Dr3Lmqpx.jpg,https://image.tmdb.org/t/p/w500/lCxz1Yus07QCQQ...


(10936, 21)


In [ ]:
final_tmdb_df.iloc[0]

,0
cache_key,movie_19995
status,success
poster_status,success
media_type,movie
source,kaggle
tmdbId,19995
title,Avatar
genres,"[Action, Adventure, Fantasy, Science Fiction]"
keywords,"[culture clash, future, space war, space colon..."
original_language,en


### **Save csv to Google Drive For Part B**

In [ ]:
final_tmdb_df.to_csv('/content/drive/MyDrive/dataset/movie/API/preprocess_dataset/final_tmdb_api.csv', index=False)

In [ ]:
ratings.to_csv('/content/drive/MyDrive/dataset/movie/API/preprocess_dataset/ratings.csv', index=False)

In [ ]:
movieLens.shape

(9739, 6)

In [ ]:
movieLens.to_csv('/content/drive/MyDrive/dataset/movie/API/preprocess_dataset/movieLens.csv', index=False)